In [1]:
"""
LLM Setup Module.
Initializes the LLM client using the internal AIA gateway.

Two approaches provided:
  APPROACH 1 (Recommended): Use LangChain's ChatOpenAI with custom base_url
             Your gateway is OpenAI-compatible, so this works directly.

  APPROACH 2 (Fallback):    Custom LangChain LLM wrapper using your
             raw llm_api() function via requests.

Set APPROACH = 1 or 2 below.
"""
import os
import json
import time
import certifi
import requests
import config
import sys
# Go 2 levels up from the current working directory
sys.path.insert(0, os.path.abspath(os.path.join('..', '..')))

import helper.auth_utils as auth_utils

APPROACH = 1  # ← Change to 2 if Approach 1 doesn't work



# ── Your internal auth modules ────────────────────────────
import helper.auth_utils as auth_utils
import helper.cert_utils as cert_utils
import helper.requests_dynamic_auth as requests_dynamic_auth
from config import base_url,TABLE_CASE_BASIC
from config import LLM_BASE_URL, LLM_MODEL, LLM_TEMPERATURE, LLM_MAX_TOKENS


# ══════════════════════════════════════════════════════════
# SHARED: Auth & Certificate Setup
# ══════════════════════════════════════════════════════════

def initialize_auth():
    """
    Initialize authentication settings, certificates,
    and dynamic auth instance. Called once at startup.
    Returns a dict with all auth components.
    """
    # 1. Load auth settings from .env
    settings = auth_utils.get_auth_settings('.env')
    print(auth_utils.describe_auth(settings))

    # 2. Ensure credentials if not using SSO
    auth_utils.ensure_credentials_if_needed(settings)

    # 3. Update certificates
    cert_utils.update_certificates()

    # 4. Build headers and http client
    default_headers = auth_utils.build_default_headers(settings)
    http_client = auth_utils.build_http_client(settings)

    # 5. Dynamic auth instance
    dynamic_auth_instance = requests_dynamic_auth.RequestsDynamicAuth()

    return {
        "settings": settings,
        "default_headers": default_headers,
        "http_client": http_client,
        "dynamic_auth_instance": dynamic_auth_instance,
    }


# ══════════════════════════════════════════════════════════
# APPROACH 1: LangChain ChatOpenAI with Custom Base URL
# ══════════════════════════════════════════════════════════

def build_llm_approach_1(auth_components: dict):
    """
    Use LangChain's ChatOpenAI pointed at your internal gateway.
    Works because your gateway exposes an OpenAI-compatible API.
    """
    from langchain_openai import ChatOpenAI

    settings = auth_components["settings"]
    default_headers = auth_components["default_headers"]
    http_client = auth_components["http_client"]

    llm = ChatOpenAI(
        base_url=LLM_BASE_URL,
        model=LLM_MODEL,
        temperature=LLM_TEMPERATURE,
        max_tokens=LLM_MAX_TOKENS,
        api_key="not-empty",  # Required by client but replaced by your auth
        default_headers=default_headers,
        http_client=http_client,
        streaming=False,
    )

    return llm


# ══════════════════════════════════════════════════════════
# APPROACH 2: Custom LangChain LLM using your llm_api()
# ══════════════════════════════════════════════════════════

def build_llm_approach_2(auth_components: dict):
    """
    Wrap your existing llm_api() function in a LangChain-compatible
    ChatModel so the agent can use it.
    """
    from typing import Any, List, Optional
    from langchain_core.language_models.chat_models import BaseChatModel
    from langchain_core.messages import (
        BaseMessage,
        AIMessage,
        HumanMessage,
        SystemMessage,
    )
    from langchain_core.outputs import ChatResult, ChatGeneration
    from langchain_core.callbacks import CallbackManagerForLLMRun

    settings = auth_components["settings"]
    default_headers = auth_components["default_headers"]
    dynamic_auth_instance = auth_components["dynamic_auth_instance"]

    class CustomGatewayChat(BaseChatModel):
        """LangChain ChatModel that calls the AIA gateway."""

        model_name: str = LLM_MODEL
        temperature: float = LLM_TEMPERATURE
        max_tokens: int = LLM_MAX_TOKENS

        @property
        def _llm_type(self) -> str:
            return "custom-aia-gateway"

        def _convert_messages(
            self, messages: List[BaseMessage]
        ) -> List[dict]:
            """Convert LangChain messages to OpenAI-format dicts."""
            converted = []
            for msg in messages:
                if isinstance(msg, SystemMessage):
                    converted.append(
                        {"role": "system", "content": msg.content}
                    )
                elif isinstance(msg, HumanMessage):
                    converted.append(
                        {"role": "user", "content": msg.content}
                    )
                elif isinstance(msg, AIMessage):
                    content = msg.content

                    # Handle tool calls if present
                    if hasattr(msg, "tool_calls") and msg.tool_calls:
                        tool_calls_formatted = []
                        for tc in msg.tool_calls:
                            tool_calls_formatted.append({
                                "id": tc.get("id", ""),
                                "type": "function",
                                "function": {
                                    "name": tc.get("name", ""),
                                    "arguments": json.dumps(
                                        tc.get("args", {})
                                    ),
                                },
                            })
                        converted.append({
                            "role": "assistant",
                            "content": content or "",
                            "tool_calls": tool_calls_formatted,
                        })
                    else:
                        converted.append(
                            {"role": "assistant", "content": content}
                        )
                else:
                    # ToolMessage, FunctionMessage, etc.
                    converted.append(
                        {"role": msg.type, "content": msg.content}
                    )
            return converted

        def _call_api(self, messages_dicts: List[dict]) -> str:
            """Make the actual API call to the gateway."""
            url = f"{LLM_BASE_URL}/chat/completions"
            headers = auth_utils.build_default_headers(settings)

            data = {
                "messages": messages_dicts,
                "temperature": self.temperature,
                "top_p": 0.95,
                "max_tokens": self.max_tokens,
                "stream": False,
                "model": self.model_name,
            }

            try:
                if settings.use_sso or settings.server_side_token_refersh:
                    response = requests.post(
                        url,
                        headers=headers,
                        json=data,
                        stream=False,
                        verify=certifi.where(),
                    )
                else:
                    response = requests.post(
                        url,
                        headers=headers,
                        json=data,
                        stream=False,
                        auth=dynamic_auth_instance,
                        verify=certifi.where(),
                    )

                response.raise_for_status()
                response_dict = response.json()

                content = (
                    response_dict["choices"][0]["message"]["content"]
                )
                return content

            except requests.exceptions.HTTPError as err:
                error_msg = (
                    f"HTTP Error {err.response.status_code}: "
                    f"{err.response.text}"
                )
                print(error_msg)
                return f"Error calling LLM: {error_msg}"
            except Exception as err:
                print(f"Error: {err}")
                return f"Error calling LLM: {err}"

        def _generate(
            self,
            messages: List[BaseMessage],
            stop: Optional[List[str]] = None,
            run_manager: Optional[CallbackManagerForLLMRun] = None,
            **kwargs: Any,
        ) -> ChatResult:
            """Required by LangChain — generate a response."""
            messages_dicts = self._convert_messages(messages)
            content = self._call_api(messages_dicts)

            message = AIMessage(content=content)
            generation = ChatGeneration(message=message)
            return ChatResult(generations=[generation])

    return CustomGatewayChat()


# ══════════════════════════════════════════════════════════
# MAIN ENTRY POINT
# ══════════════════════════════════════════════════════════

def build_llm():
    """
    Initialize auth, build, and return the LLM instance.
    Uses APPROACH 1 or 2 based on the setting at top of file.
    """
    print(f"Initializing LLM (Approach {APPROACH})...")
    print(f"Model: {LLM_MODEL}")
    print(f"Gateway: {LLM_BASE_URL}")

    auth_components = initialize_auth()

    if APPROACH == 1:
        llm = build_llm_approach_1(auth_components)
    else:
        llm = build_llm_approach_2(auth_components)

    print("✅ LLM initialized successfully.")
    return llm, auth_components

Using Client Credentials

=== TOKEN SCOPES ===
ddc-api.ddc-api-scope
aia-gateway.ground-truth
aia-gateway.genai.dev.mcps
aia-gateway.aidaas.dev
aia-gateway.prompt-eval
aia-gateway.genai.gdc-dev
aia-gateway.llmj
aia-gateway.genai.gcp-dev
aia-gateway.genai.dev
aia-gateway.genai.dev.batch
aia-gateway.prompt-gen
Downloaded certificate zip, size: 14933 bytes
Certifi bundle path: C:\Users\Prabal_Paul\AppData\Roaming\Python\Python312\site-packages\certifi\cacert.pem
Appending Dell certificates to certifi bundle...
Dell certificates successfully added to certifi bundle.
Dell certificate utilities loaded (no certificates appended by default).


In [3]:
%pip install langchain-openai

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_openai-1.3.4-py3-none-any.whl.metadata (3.4 kB)
  Using cached langchain_core-1.4.9-py3-none-any.whl.metadata (4.7 kB)
Using cached langchain_core-1.4.9-py3-none-any.whl (558 kB)

  Attempting uninstall: langchain-core

    Found existing installation: langchain-core 0.3.86

    Uninstalling langchain-core-0.3.86:

      Successfully uninstalled langchain-core-0.3.86

   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---------------------------------------- 0/2 [langchain-core]
   ---

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain 0.3.26 requires langchain-core<1.0.0,>=0.3.66, but you have langchain-core 1.4.9 which is incompatible.


In [4]:
%pip install "langchain-core==0.3.67"
%pip install "langchain-openai==0.3.17"

Defaulting to user installation because normal site-packages is not writeable
  Using cached langchain_core-0.3.67-py3-none-any.whl.metadata (5.8 kB)
Using cached langchain_core-0.3.67-py3-none-any.whl (440 kB)
Note: you may need to restart the kernel to use updated packages.


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-text-splitters 0.3.11 requires langchain-core<2.0.0,>=0.3.75, but you have langchain-core 0.3.67 which is incompatible.
langgraph-prebuilt 1.1.0 requires langchain-core>=1.3.1, but you have langchain-core 0.3.67 which is incompatible.
langgraph-sdk 0.4.2 requires langchain-core<2,>=1.4.0, but you have langchain-core 0.3.67 which is incompatible.


Defaulting to user installation because normal site-packages is not writeable
   ---------------------------------------- 0.0/948.6 kB ? eta -:--:--
   ---------------------------------------- 0.0/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   ----------- ---------------------------- 262.1/948.6 kB ? eta -:--:--
   --------------------- ---------------- 524.3/94

In [5]:
!pip list | findstr /I "langchain"

langchain                 0.3.26
langchain-core            0.3.67
langchain-openai          0.3.17
langchain-protocol        0.0.18
langchain-text-splitters  0.3.11
